# Session 3 — Pixel Losses and Honest Scores 🧪

**Time:** 35–40 minutes  
**Goal:** score a segmentation prediction while excluding ignored border pixels.

You will work with tiny masks where every count is visible, then use the same accumulation pattern used for a validation set.

## 0. Setup

This lab has no download and no training step. Run the cells in order.

In [23]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
IGNORE_INDEX = 255
CLASS_NAMES = ['background', 'pet']
print('PyTorch:', torch.__version__)

PyTorch: 2.11.0+cpu


## 1. From logits to a mask (5 min)

A model gives two **logits** per pixel: one for background and one for pet. Softmax turns each pair into probabilities that sum to one; `argmax` selects the larger probability.

In [24]:
# Shape: [batch, classes, height, width]
logits = torch.tensor([[[[3.0, -1.0], [0.2, 1.2]],
                        [[-1.0, 3.0], [1.5, 0.3]]]])
probabilities = logits.softmax(dim=1)
prediction = probabilities.argmax(dim=1)

print('logits shape:       ', tuple(logits.shape))
print('probability sums:\n', probabilities.sum(dim=1))
print('prediction shape:   ', tuple(prediction.shape))
print('predicted labels:\n', prediction[0])
assert torch.allclose(probabilities.sum(dim=1), torch.ones_like(probabilities[:, 0]))
assert prediction.shape == (1, 2, 2)

logits shape:        (1, 2, 2, 2)
probability sums:
 tensor([[[1., 1.],
         [1., 1.]]])
prediction shape:    (1, 2, 2)
predicted labels:
 tensor([[0, 1],
        [1, 0]])


## 2. Ignored borders do not vote (6 min)

Oxford-IIIT Pet trimaps use `255` for an ambiguous border. It is neither background nor pet, so it must be excluded from both the loss and every metric.

In [25]:
targets = torch.tensor([[[0, 1], [IGNORE_INDEX, 0]]])
valid = targets != IGNORE_INDEX
loss = F.cross_entropy(logits, targets, ignore_index=IGNORE_INDEX)

print('valid pixels:', int(valid.sum()), 'of', targets.numel())
print('valid mask:\n', valid[0])
print(f'cross-entropy on valid pixels only: {loss.item():.3f}')
assert int(valid.sum()) == 3

valid pixels: 3 of 4
valid mask:
 tensor([[ True,  True],
        [False,  True]])
cross-entropy on valid pixels only: 0.126


## 3. Exercise E1 — accumulate only valid pixels (10 min)

Complete the helper below. It must update a 2×2 confusion matrix whose **rows are true classes** and **columns are predicted classes**. Ignore every target equal to `255`.

In [26]:
# TODO: update the confusion matrix using only target != IGNORE_INDEX pixels
def update_confusion(confusion: torch.Tensor, target: torch.Tensor, predicted: torch.Tensor) -> torch.Tensor:
    """
    Update a 2x2 confusion matrix (rows=true, cols=pred) ignoring IGNORE_INDEX pixels.
    Assumes classes are {0,1} and confusion is dtype torch.int64.
    """
    # Aplanar tensores
    t = target.view(-1)
    p = predicted.view(-1)

    # Máscara de píxeles válidos
    valid_mask = t != IGNORE_INDEX
    if valid_mask.sum() == 0:
        return confusion

    t_valid = t[valid_mask]
    p_valid = p[valid_mask]

    # Mapear pares (true,pred) a índices 0..3: idx = true*2 + pred
    idx = (t_valid * 2 + p_valid).to(torch.long)

    # Contar ocurrencias para cada par; asegurar longitud mínima 4
    counts = torch.bincount(idx, minlength=4).to(torch.int64)

    # Reshape a 2x2 y sumar a la confusión existente
    counts_2x2 = counts.view(2, 2)
    confusion = confusion + counts_2x2

    return confusion


In [27]:
def scores_from_confusion(confusion: torch.Tensor):
    """
    Calculates pixel accuracy and class-wise Intersection over Union (IoU) from a confusion matrix.
    Args:
        confusion: A 2x2 torch.Tensor representing the confusion matrix.
                   confusion[i, j] is the count of pixels that are truly class i and predicted as class j.
    Returns:
        tuple: (pixel_accuracy, class_iou)
            pixel_accuracy (float): The overall pixel accuracy.
            class_iou (torch.Tensor): A 1D tensor containing IoU for each class.
    """
    # Total number of correctly classified pixels
    correct_pixels = torch.diag(confusion).sum().item()
    # Total number of pixels
    total_pixels = confusion.sum().item()

    # Pixel Accuracy
    pixel_accuracy = correct_pixels / total_pixels if total_pixels > 0 else 0.0

    # Class-wise IoU
    class_iou = torch.zeros(confusion.shape[0], dtype=torch.float32)
    for k in range(confusion.shape[0]):
        # True Positives for class k
        tp_k = confusion[k, k].item()
        # False Positives for class k (pixels predicted as k, but are not k)
        fp_k = confusion[:, k].sum().item() - tp_k
        # False Negatives for class k (pixels that are k, but not predicted as k)
        fn_k = confusion[k, :].sum().item() - tp_k

        union_k = tp_k + fp_k + fn_k
        class_iou[k] = tp_k / union_k if union_k > 0 else 0.0

    return pixel_accuracy, class_iou

# Two small saved predictions: accumulate counts first, then score once.
batch_targets = [
    torch.tensor([[0, 1], [1, IGNORE_INDEX]]),
    torch.tensor([[0, 0], [1, 1]]),
]
batch_predictions = [
    torch.tensor([[0, 1], [0, 1]]),
    torch.tensor([[0, 1], [1, 0]]),
]

confusion = torch.zeros((2, 2), dtype=torch.int64)
for target, predicted in zip(batch_targets, batch_predictions):
    confusion = update_confusion(confusion, target, predicted)

accuracy, class_iou = scores_from_confusion(confusion)
print('rows=true, columns=predicted')
print(confusion)
print(f'dataset pixel accuracy: {accuracy:.3f}')
print(f'background IoU: {class_iou[0]:.3f}; pet IoU: {class_iou[1]:.3f}')
assert confusion.sum().item() == 7  # one of eight cells was ignored
assert torch.all((class_iou >= 0) & (class_iou <= 1))

rows=true, columns=predicted
tensor([[2, 1],
        [2, 2]])
dataset pixel accuracy: 0.571
background IoU: 0.400; pet IoU: 0.400


**Why accumulate?** Averaging IoU separately for each image gives every image equal weight, even if one has many more valid pixels. A validation report should state its aggregation rule; here we total counts over the dataset before calculating the score.

## 4. Exercise E2 — the worked 4×4 pet IoU (8 min)

For the pet class, find `TP`, `FP`, and `FN`. The ignored lower-right cell must disappear before counting. The expected result is $3/(3+1+1)=0.60$.

In [28]:
target_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, IGNORE_INDEX],
])
predicted_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 1],
])

In [29]:
# TODO: calculate the pet TP, FP, FN, and IoU while excluding ignored labels
# Asumiendo IGNORE_INDEX ya definido (255)
target_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, IGNORE_INDEX],
])
predicted_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 1],
])

# Máscara de píxeles válidos (excluir IGNORE_INDEX)
valid = target_4x4 != IGNORE_INDEX
t = target_4x4[valid]
p = predicted_4x4[valid]

# Clase objetivo: pet = 1
cls = 1

# True Positives: target==1 and pred==1
tp = int(((t == cls) & (p == cls)).sum().item())

# False Positives: target!=1 but pred==1
fp = int(((t != cls) & (p == cls)).sum().item())

# False Negatives: target==1 but pred!=1
fn = int(((t == cls) & (p != cls)).sum().item())

# IoU
den = tp + fp + fn
iou_pet = tp / den if den > 0 else 0.0

print(f"TP={tp}, FP={fp}, FN={fn}, IoU_pet={iou_pet:.2f}")

# Comprobación esperada
assert tp == 3 and fp == 1 and fn == 1
assert abs(iou_pet - 0.6) < 1e-6

TP=3, FP=1, FN=1, IoU_pet=0.60


## 5. Accuracy can hide a bad foreground mask (5 min)

Suppose a 10×10 image contains only four pet pixels. A model predicts every pixel as background. It is correct on 96 of 100 pixels, but it finds none of the pet.

In [30]:
rare_target = torch.zeros((10, 10), dtype=torch.int64)
rare_target[:2, :2] = 1
all_background = torch.zeros_like(rare_target)

rare_confusion = torch.zeros((2, 2), dtype=torch.int64)
rare_confusion = update_confusion(rare_confusion, rare_target, all_background)
rare_accuracy, rare_iou = scores_from_confusion(rare_confusion)
print(f'pixel accuracy: {rare_accuracy:.2%}')
print(f'pet IoU: {rare_iou[1]:.2f}')
assert rare_accuracy == 0.96 and rare_iou[1] == 0.0

pixel accuracy: 96.00%
pet IoU: 0.00


> TODO: In one or two sentences, explain why the 96% pixel accuracy above is misleading.


## Checkpoint

Before leaving, confirm that you can explain: (1) why `255` is excluded, (2) why the 4×4 pet IoU is 0.60, and (3) why a dataset metric starts from accumulated counts. Next session you will use these metrics to evaluate a pretrained segmentation model.